# 02-2. Обработка моделей тональности и сборка финальных выходов

Этот ноутбук отвечает за модельный этап: выбор текстового поля, загрузку или использование готовых результатов восьми моделей тональности, агрегацию на дневном уровне и сборку итоговой таблицы. По умолчанию пересборка отключена, чтобы не перезапускать модели и не скачивать ресурсы повторно

Входы из ноутбука `02-1_data_before_models.ipynb`:
- `outputs_01/returns_targets_2019_2023.parquet`
- `outputs_01/trade_calendar_2019_2023.parquet`

Основные выходы:
- `outputs_01/text_variant_finbert_report.csv`
- `outputs_01/selected_text_variants.json`
- `outputs_01/daily_news_*.parquet`
- `outputs_01/returns_sentiment_enhanced.parquet`
- `master_sample_TSLA_oct2023.csv`

## Блок 1. Импорт библиотек

In [2]:
import os
import gc
import json
import warnings
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm
from pandas.util import hash_pandas_object

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from textblob import TextBlob

import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

from IPython.display import display

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 200)

## Блок 2. Конфигурация проекта

Задаём пути и фиксируем, что пересборка моделей по умолчанию отключена

In [3]:
START_DATE = pd.Timestamp("2019-01-01")
END_DATE = pd.Timestamp("2023-12-31")

OUTPUT_DIR = "outputs_01"

NEWS_SOURCES = {
    "nasdaq": os.path.join(OUTPUT_DIR, "news_deduped_2019_2023.csv"),
}

SELECTED_EQUITIES_PATH = os.path.join(OUTPUT_DIR, "selected_equities_only_2019_2023.csv")
RETURNS_TARGETS_PATH = os.path.join(OUTPUT_DIR, "returns_targets_2019_2023.parquet")
TRADE_CALENDAR_PATH = os.path.join(OUTPUT_DIR, "trade_calendar_2019_2023.parquet")
TEXT_VARIANT_REPORT_PATH = os.path.join(OUTPUT_DIR, "text_variant_finbert_report.csv")
SELECTED_VARIANTS_PATH = os.path.join(OUTPUT_DIR, "selected_text_variants.json")

CHUNK_SIZE = 200000
MAX_NEWS_PER_DAY = None

ALLOW_TEXT_VARIANT_REBUILD = False
FORCE_REBUILD_TEXT_VARIANT = False
ALLOW_MODEL_REBUILD = False

RESUME_FROM_CHECKPOINTS = True
FORCE_RECOMPUTE_CHECKPOINTS = False
SKIP_IF_DAILY_EXISTS = True
FORCE_REBUILD_DAILY = False

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

HF_MAX_LENGTH = 512
HF_BATCH_SIZE = 32

base_tag = "allnews" if (MAX_NEWS_PER_DAY is None or int(MAX_NEWS_PER_DAY) <= 0) else f"max{int(MAX_NEWS_PER_DAY)}"
RUN_TAG = f"{base_tag}__len{int(HF_MAX_LENGTH)}"

os.makedirs(OUTPUT_DIR, exist_ok=True)

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints", RUN_TAG)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("Устройство:", DEVICE)
print("Папка вывода:", OUTPUT_DIR)
print("Папка чекпойнтов:", CHECKPOINT_DIR)
print("Пересборка вариантов текста разрешена:", ALLOW_TEXT_VARIANT_REBUILD)
print("Пересборка модельных выходов разрешена:", ALLOW_MODEL_REBUILD)

Устройство: mps
Папка вывода: outputs_01
Папка чекпойнтов: outputs_01/checkpoints/allnews__len512
Пересборка вариантов текста разрешена: False
Пересборка модельных выходов разрешена: False


## Блок 3. Загрузка подготовленных данных

Проверяем входные файлы и загружаем таблицы, подготовленные на предыдущем этапе

In [4]:
if not os.path.exists(SELECTED_EQUITIES_PATH):
    raise FileNotFoundError(f"Файл со списком тикеров не найден: {SELECTED_EQUITIES_PATH}")

missing_news = [fp for fp in NEWS_SOURCES.values() if not os.path.exists(fp)]
if missing_news:
    raise FileNotFoundError(f"Файл(ы) с новостями не найден(ы): {missing_news}")

missing_prepared = [
    fp for fp in [RETURNS_TARGETS_PATH, TRADE_CALENDAR_PATH]
    if not os.path.exists(fp)
]
if missing_prepared:
    raise FileNotFoundError(
        "Не найдены промежуточные файлы из ноутбука 02-1: "
        + ", ".join(missing_prepared)
    )

companies = pd.read_csv(SELECTED_EQUITIES_PATH, low_memory=False)
companies["ticker"] = companies["ticker"].astype(str).str.strip()
TICKER_SET = set(companies["ticker"].tolist())

ret = pd.read_parquet(RETURNS_TARGETS_PATH)
ret["ticker"] = ret["ticker"].astype(str).str.strip()
ret["date"] = pd.to_datetime(ret["date"], errors="coerce").dt.normalize()
ret = ret.dropna(subset=["ticker", "date"]).sort_values(["ticker", "date"], kind="mergesort").reset_index(drop=True)

trade = pd.read_parquet(TRADE_CALENDAR_PATH)
trade["ticker"] = trade["ticker"].astype(str).str.strip()
trade["trade_date"] = pd.to_datetime(trade["trade_date"], errors="coerce").dt.normalize()
trade = trade.dropna(subset=["ticker", "trade_date"]).sort_values(["ticker", "trade_date"], kind="mergesort").reset_index(drop=True)

print("Тикеров в выборке:", len(TICKER_SET))
print("Размер таблицы доходностей и целей:", ret.shape)
print("Размер торгового календаря:", trade.shape)
display(companies.head(5))

Тикеров в выборке: 2054
Размер таблицы доходностей и целей: (2580236, 12)
Размер торгового календаря: (2580236, 2)


,ticker,company_name,sector,industry,exchange,quoteType,price_rows,price_min_date,price_max_date,coverage_pct,starts_in_2019,ends_in_2023,full_coverage,news_count,news_min_date,news_max_date
0,TSLA,"Tesla, Inc.",Consumer Cyclical,Auto Manufacturers,NMS,EQUITY,1257,2019-01-02,2023-12-28,100.0,True,True,True,10587,2019-07-01,2023-12-16
1,GOOG,Alphabet Inc.,Communication Services,Internet Content & Information,NMS,EQUITY,1257,2019-01-02,2023-12-28,100.0,True,True,True,9883,2019-01-02,2023-12-16
2,DIS,The Walt Disney Company,Communication Services,Entertainment,NYQ,EQUITY,1257,2019-01-02,2023-12-28,100.0,True,True,True,9654,2019-06-13,2023-12-16
3,BHFAL,"Brighthouse Financial, Inc.",NaN,NaN,NMS,EQUITY,1257,2019-01-02,2023-12-28,100.0,True,True,True,9614,2023-11-30,2023-12-16
4,AAPL,Apple Inc.,Technology,Consumer Electronics,NMS,EQUITY,1257,2019-01-02,2023-12-28,100.0,True,True,True,9338,2020-03-09,2023-12-16


## Блок 4. Подготовка текстов

Определяем функции для нормализации новостей и сборки текстовых вариантов

In [5]:
NEWS_COLS_RAW = [
    "Date", "Article_title", "Stock_symbol", "Url", "Publisher", "Author",
    "Article", "Lsa_summary", "Luhn_summary", "Textrank_summary", "Lexrank_summary",
]

def print_step(title: str):
    line = "=" * 88
    print("\n" + line)
    print(title)
    print(line)

def _clean_str(s: pd.Series) -> pd.Series:
    return s.fillna("").astype(str).str.strip()

def _normalize_news_cols(df: pd.DataFrame) -> pd.DataFrame:
    ren = {}
    if "Date" in df.columns and "date" not in df.columns:
        ren["Date"] = "date"
    if "Stock_symbol" in df.columns and "ticker" not in df.columns:
        ren["Stock_symbol"] = "ticker"
    if "Url" in df.columns and "url" not in df.columns:
        ren["Url"] = "url"
    if "Article_title" in df.columns and "title" not in df.columns:
        ren["Article_title"] = "title"
    if "Article" in df.columns and "article" not in df.columns:
        ren["Article"] = "article"
    if "Lsa_summary" in df.columns and "lsa_summary" not in df.columns:
        ren["Lsa_summary"] = "lsa_summary"
    if "Luhn_summary" in df.columns and "luhn_summary" not in df.columns:
        ren["Luhn_summary"] = "luhn_summary"
    if "Textrank_summary" in df.columns and "textrank_summary" not in df.columns:
        ren["Textrank_summary"] = "textrank_summary"
    if "Lexrank_summary" in df.columns and "lexrank_summary" not in df.columns:
        ren["Lexrank_summary"] = "lexrank_summary"
    if "Publisher" in df.columns and "publisher" not in df.columns:
        ren["Publisher"] = "publisher"
    if "Author" in df.columns and "author" not in df.columns:
        ren["Author"] = "author"
    return df.rename(columns=ren) if ren else df

def _ensure_cols(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    df = df.copy()
    for c in cols:
        if c not in df.columns:
            df[c] = np.nan
    return df

def _prep_news_chunk_base(raw: pd.DataFrame, source_tag: str) -> pd.DataFrame:
    ch = _normalize_news_cols(raw)

    need = [
        "date", "ticker", "url", "title", "article",
        "lsa_summary", "luhn_summary", "textrank_summary", "lexrank_summary",
        "publisher", "author"
    ]
    ch = _ensure_cols(ch, need)

    ch["ticker"] = _clean_str(ch["ticker"])
    ch["url"] = _clean_str(ch["url"])

    ch = ch[ch["ticker"] != ""]
    ch = ch[ch["url"] != ""]
    if ch.empty:
        return ch.iloc[0:0].copy()

    ch["date"] = pd.to_datetime(ch["date"], errors="coerce", utc=True).dt.tz_convert(None)
    ch = ch.dropna(subset=["date"])
    if ch.empty:
        return ch.iloc[0:0].copy()

    ch["day"] = ch["date"].dt.floor("D")
    ch = ch[(ch["day"] >= START_DATE) & (ch["day"] <= END_DATE)]
    if ch.empty:
        return ch.iloc[0:0].copy()

    ch = ch[ch["ticker"].isin(TICKER_SET)]
    if ch.empty:
        return ch.iloc[0:0].copy()

    if MAX_NEWS_PER_DAY is not None and int(MAX_NEWS_PER_DAY) > 0:
        ch = (
            ch.sort_values(["ticker", "day"], kind="mergesort")
              .groupby(["ticker", "day"], as_index=False, sort=False)
              .head(int(MAX_NEWS_PER_DAY))
        )

    ch = ch.drop_duplicates(subset=["ticker", "day", "url"])
    ch["source"] = source_tag

    keep_cols = [
        "ticker", "day", "url",
        "title", "article",
        "lsa_summary", "luhn_summary", "textrank_summary", "lexrank_summary",
        "publisher", "author",
        "source"
    ]
    return ch[keep_cols].reset_index(drop=True)

def build_text_variant(df: pd.DataFrame, variant: str) -> pd.Series:
    title = _clean_str(df["title"]) if "title" in df.columns else pd.Series([""] * len(df), index=df.index)
    article = _clean_str(df["article"]) if "article" in df.columns else pd.Series([""] * len(df), index=df.index)

    lsa = _clean_str(df["lsa_summary"]) if "lsa_summary" in df.columns else pd.Series([""] * len(df), index=df.index)
    luhn = _clean_str(df["luhn_summary"]) if "luhn_summary" in df.columns else pd.Series([""] * len(df), index=df.index)
    tr = _clean_str(df["textrank_summary"]) if "textrank_summary" in df.columns else pd.Series([""] * len(df), index=df.index)
    lex = _clean_str(df["lexrank_summary"]) if "lexrank_summary" in df.columns else pd.Series([""] * len(df), index=df.index)

    if variant == "title_article":
        comb = (title + ". " + article).str.replace(r"\s+", " ", regex=True).str.strip()
        out = pd.Series([""] * len(df), index=df.index, dtype="string")
        both = (title != "") & (article != "")
        only_t = (title != "") & (article == "")
        only_a = (title == "") & (article != "")
        out.loc[both] = comb.loc[both]
        out.loc[only_t] = title.loc[only_t]
        out.loc[only_a] = article.loc[only_a]
        return out.astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

    if variant == "lsa_summary":
        return lsa.where(lsa != "", "")
    if variant == "luhn_summary":
        return luhn.where(luhn != "", "")
    if variant == "textrank_summary":
        return tr.where(tr != "", "")
    if variant == "lexrank_summary":
        return lex.where(lex != "", "")

    raise ValueError(f"Неизвестный вариант текста: {variant}")

## Блок 5. Отчёт по вариантам текста

In [6]:
TEXT_VARIANTS = ["title_article", "lsa_summary", "luhn_summary", "textrank_summary", "lexrank_summary"]

FINBERT_EVAL_SAMPLE_MOD = 500

def _load_hf_model(model_id: str):
    tok = AutoTokenizer.from_pretrained(model_id, use_fast=True, local_files_only=True)
    mdl = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if DEVICE in ("mps", "cpu") else torch.float32,
        local_files_only=True
    ).to(DEVICE)
    mdl.eval()
    raw = getattr(mdl.config, "id2label", {}) or {}
    id2lab = {int(k): str(v).lower() for k, v in raw.items()} if isinstance(raw, dict) else {}
    raw2 = getattr(mdl.config, "label2id", {}) or {}
    lab2id = {str(k).lower(): int(v) for k, v in raw2.items()} if isinstance(raw2, dict) else {}
    return tok, mdl, id2lab, lab2id

def _resolve_sentiment_slots(id2lab: dict, lab2id: dict):
    def pick_from_id2lab(keys):
        for i, lab in id2lab.items():
            lab2 = str(lab).lower()
            for k in keys:
                if k in lab2:
                    return int(i)
        return None

    def pick_from_lab2id(keys):
        for k in keys:
            for lab, i in lab2id.items():
                if k in str(lab).lower():
                    return int(i)
        return None

    neg = pick_from_id2lab(["negative", "neg", "bearish"])
    pos = pick_from_id2lab(["positive", "pos", "bullish"])
    neu = pick_from_id2lab(["neutral", "neu", "none"])

    if neg is None:
        neg = pick_from_lab2id(["negative", "neg", "bearish"])
    if pos is None:
        pos = pick_from_lab2id(["positive", "pos", "bullish"])
    if neu is None:
        neu = pick_from_lab2id(["neutral", "neu", "none"])

    return neg, neu, pos

def _hf_predict(texts: List[str], tok, mdl, neg_i, neu_i, pos_i,
                batch_size: int = None, max_length: int = None):
    batch_size = int(HF_BATCH_SIZE if batch_size is None else batch_size)
    max_length = int(HF_MAX_LENGTH if max_length is None else max_length)

    n = len(texts)
    pneg = np.zeros(n, dtype=np.float32)
    pneu = np.zeros(n, dtype=np.float32)
    ppos = np.zeros(n, dtype=np.float32)

    with torch.inference_mode():
        for a in range(0, n, batch_size):
            b = texts[a:a + batch_size]
            enc = tok(
                b,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            logits = mdl(**enc).logits
            pr = torch.softmax(logits, dim=-1).detach().cpu().numpy().astype(np.float32)

            if neg_i is None or pos_i is None:
                if pr.shape[1] == 2:
                    pneg[a:a + len(b)] = pr[:, 0]
                    ppos[a:a + len(b)] = pr[:, 1]
                    pneu[a:a + len(b)] = 0.0
                else:
                    raise ValueError("Не удалось определить индексы классов тональности (neg/pos) для этой модели")
            else:
                pneg[a:a + len(b)] = pr[:, int(neg_i)]
                ppos[a:a + len(b)] = pr[:, int(pos_i)]
                pneu[a:a + len(b)] = pr[:, int(neu_i)] if neu_i is not None else 0.0

    score = (ppos - pneg).astype(np.float32)
    abs_score = np.abs(score).astype(np.float32)
    return score, abs_score, pneg, pneu, ppos

def _sample_for_eval(ch: pd.DataFrame) -> pd.DataFrame:
    m = FINBERT_EVAL_SAMPLE_MOD
    if m is None or int(m) <= 1:
        return ch
    url = _clean_str(ch["url"]).str.lower()
    ticker = _clean_str(ch["ticker"])
    key = (url + "||" + ticker).values
    h = hash_pandas_object(pd.Series(key), index=False).astype("uint64").to_numpy()
    keep = (h % int(m)) == 0
    return ch.loc[keep].reset_index(drop=True)

def eval_text_variants_finbert(source_tag: str, file_path: str,
                              batch_size: int = None, max_length: int = None,
                              model_id: str = "ProsusAI/finbert") -> pd.DataFrame:
    print_step(f"Оценка вариантов текста с помощью FinBERT: источник={source_tag}")

    tok, mdl, id2lab, lab2id = _load_hf_model(model_id)
    neg_i, neu_i, pos_i = _resolve_sentiment_slots(id2lab, lab2id)

    metrics = {v: {"rows_after_filters": 0, "rows_with_text": 0,
                   "len_sum": 0, "neu_sum": 0.0, "abs_score_sum": 0.0}
               for v in TEXT_VARIANTS}

    reader = pd.read_csv(file_path, chunksize=CHUNK_SIZE, low_memory=False)

    for raw in tqdm(reader, desc=f"FinBERT scan: {source_tag}"):
        ch = _prep_news_chunk_base(raw, source_tag)
        if ch.empty:
            continue

        ch = _sample_for_eval(ch)
        if ch.empty:
            continue

        nrows = int(len(ch))
        for v in TEXT_VARIANTS:
            metrics[v]["rows_after_filters"] += nrows

        for v in TEXT_VARIANTS:
            text = build_text_variant(ch, v)
            m = (text != "")
            if not m.any():
                continue

            texts = text.loc[m].tolist()
            metrics[v]["rows_with_text"] += len(texts)
            metrics[v]["len_sum"] += int(pd.Series(texts).str.len().sum())

            score, abs_score, pneg, pneu, ppos = _hf_predict(
                texts, tok, mdl, neg_i, neu_i, pos_i,
                batch_size=batch_size, max_length=max_length
            )

            metrics[v]["neu_sum"] += float(np.sum(pneu))
            metrics[v]["abs_score_sum"] += float(np.sum(abs_score))

    rows = []
    for v in TEXT_VARIANTS:
        a = metrics[v]["rows_after_filters"]
        n = metrics[v]["rows_with_text"]

        coverage = (n / a) if a else 0.0
        avg_len = (metrics[v]["len_sum"] / n) if n else 0.0
        avg_neu = (metrics[v]["neu_sum"] / n) if n else 1.0
        non_neutral = 1.0 - avg_neu
        mean_abs_score = (metrics[v]["abs_score_sum"] / n) if n else 0.0

        score = coverage * non_neutral * mean_abs_score

        rows.append({
            "source": source_tag,
            "variant": v,
            "rows_after_filters": int(a),
            "rows_with_text": int(n),
            "coverage": float(coverage),
            "avg_len_chars": float(avg_len),
            "avg_neu_share": float(avg_neu),
            "non_neutral": float(non_neutral),
            "mean_abs_score": float(mean_abs_score),
            "composite_score": float(score),
        })

    out = pd.DataFrame(rows).sort_values("composite_score", ascending=False).reset_index(drop=True)

    del mdl
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out



report_path = TEXT_VARIANT_REPORT_PATH

if os.path.exists(report_path) and not FORCE_REBUILD_TEXT_VARIANT:
    variant_report = pd.read_csv(report_path)
    print("Загружен готовый отчет по вариантам текста:", report_path)
else:
    if not ALLOW_TEXT_VARIANT_REBUILD:
        raise FileNotFoundError(
            f"Готовый отчет не найден: {report_path}. Пересборка отключена, чтобы не перезапускать модели."
        )

    variant_reports = []
    for tag, fp in NEWS_SOURCES.items():
        rep = eval_text_variants_finbert(
            source_tag=tag,
            file_path=fp,
            batch_size=HF_BATCH_SIZE,
            max_length=HF_MAX_LENGTH,
            model_id="ProsusAI/finbert"
        )
        variant_reports.append(rep)
        print("Лучшие варианты для", tag)
        display(rep.head(10))

    variant_report = pd.concat(variant_reports, ignore_index=True)
    variant_report.to_csv(report_path, index=False)
    print("Сохранено:", report_path)

display(variant_report.sort_values(["source", "composite_score"], ascending=[True, False]).reset_index(drop=True))

Загружен готовый отчет по вариантам текста: outputs_01/text_variant_finbert_report.csv


,source,variant,rows_after_filters,rows_with_text,coverage,avg_len_chars,avg_neu_share,non_neutral,mean_abs_score,composite_score
0,nasdaq,title_article,2448,2448,1.000000,3865.002859,0.457470,0.542530,0.435194,0.236106
1,nasdaq,lsa_summary,2448,1800,0.735294,565.682778,0.401319,0.598681,0.504724,0.222183
2,nasdaq,lexrank_summary,2448,1800,0.735294,511.657222,0.430894,0.569106,0.478337,0.200165
3,nasdaq,luhn_summary,2448,1800,0.735294,577.751111,0.452821,0.547179,0.456088,0.183502
4,nasdaq,textrank_summary,2448,1800,0.735294,607.982778,0.461638,0.538362,0.454547,0.179935


## Блок 6. Выбор текстового поля

Выбираем лучший вариант текста для каждого источника и сохраняем метаданные

In [7]:
variant_report = pd.read_csv(TEXT_VARIANT_REPORT_PATH)

selected_variants = {}
for src in variant_report["source"].unique():
    top = (
        variant_report.loc[variant_report["source"] == src]
        .sort_values("composite_score", ascending=False)
        .iloc[0]
    )
    selected_variants[src] = str(top["variant"])

print("Выбранные варианты текста по источникам:")
print(json.dumps(selected_variants, ensure_ascii=False, indent=2))

with open(SELECTED_VARIANTS_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            "selected_variants": selected_variants,
            "text_variants_considered": TEXT_VARIANTS,
            "date_range": [str(START_DATE.date()), str(END_DATE.date())],
            "hf_max_length": int(HF_MAX_LENGTH),
            "hf_batch_size": int(HF_BATCH_SIZE),
            "finbert_eval_sample_mod": int(FINBERT_EVAL_SAMPLE_MOD),
            "max_news_per_day": None if (MAX_NEWS_PER_DAY is None or int(MAX_NEWS_PER_DAY) <= 0) else int(MAX_NEWS_PER_DAY),
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Сохранено:", SELECTED_VARIANTS_PATH)


Выбранные варианты текста по источникам:
{
  "nasdaq": "title_article"
}
Сохранено: outputs_01/selected_text_variants.json


## Блок 7. Функции для агрегации моделей

Определяем функции, которые строят дневные агрегаты тональности по каждой модели

In [8]:
with open(SELECTED_VARIANTS_PATH, "r", encoding="utf-8") as f:
    sel_meta = json.load(f)

selected_variants = sel_meta["selected_variants"]

print("Выбранные варианты:", selected_variants)

def _checkpoint_path(prefix: str, source_tag: str, variant: str) -> str:
    return os.path.join(CHECKPOINT_DIR, f"{prefix}__{source_tag}__{variant}__rawagg.parquet")

def _reduce_parts(parts: List[pd.DataFrame]) -> pd.DataFrame:
    if not parts:
        return pd.DataFrame()
    df = pd.concat(parts, ignore_index=True)
    df["ticker"] = df["ticker"].astype(str).str.strip()
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
    df = df.dropna(subset=["ticker", "date"])
    df = df.groupby(["ticker", "date"], sort=False).sum(numeric_only=True).reset_index()
    return df.sort_values(["ticker", "date"], kind="mergesort").reset_index(drop=True)

def _prep_news_chunk_variant(raw: pd.DataFrame, source_tag: str, variant: str) -> pd.DataFrame:
    ch = _prep_news_chunk_base(raw, source_tag)
    if ch.empty:
        return ch.iloc[0:0].copy()

    text = build_text_variant(ch, variant)
    m = (text != "")
    if not m.any():
        return ch.iloc[0:0].copy()

    out = ch.loc[m, ["ticker", "day"]].copy()
    out["text"] = text.loc[m].astype(str)
    return out.reset_index(drop=True)

def _vader_scorer():
    try:
        return SentimentIntensityAnalyzer()
    except Exception as e:
        raise RuntimeError(
            "Лексикон VADER не найден в локальном окружении. Автозагрузка отключена."
        ) from e

def _vader_file_rawagg(file_path: str, source_tag: str, variant: str,
                       band: float, sia: SentimentIntensityAnalyzer) -> pd.DataFrame:
    parts = []
    buffer = []
    reader = pd.read_csv(file_path, chunksize=CHUNK_SIZE, low_memory=False)

    for raw in tqdm(reader, desc=f"vader: {source_tag}"):
        ch = _prep_news_chunk_variant(raw, source_tag, variant)
        if ch.empty:
            continue

        texts = ch["text"].astype(str).tolist()
        scores = np.fromiter((float(sia.polarity_scores(x)["compound"]) for x in texts), dtype=np.float32, count=len(texts))
        neg = (scores <= -float(band)).astype(np.int32)
        pos = (scores >= float(band)).astype(np.int32)

        tmp = ch[["ticker", "day"]].copy()
        tmp = tmp.rename(columns={"day": "date"})
        tmp["n"] = 1
        tmp["ssum"] = scores.astype(np.float32)
        tmp["nneg"] = neg
        tmp["npos"] = pos

        g = tmp.groupby(["ticker", "date"], sort=False).sum(numeric_only=True).reset_index()
        buffer.append(g)

        if len(buffer) >= 25:
            parts.append(_reduce_parts(buffer))
            buffer = []

    if buffer:
        parts.append(_reduce_parts(buffer))

    out = _reduce_parts(parts)
    return out

def build_vader_daily(band: float = 0.05) -> pd.DataFrame:
    sia = _vader_scorer()

    parts = []
    for tag, fp in NEWS_SOURCES.items():
        variant = selected_variants[tag]

        cp = _checkpoint_path("vader", tag, variant)
        if RESUME_FROM_CHECKPOINTS and os.path.exists(cp) and not FORCE_RECOMPUTE_CHECKPOINTS:
            part = pd.read_parquet(cp)
        else:
            part = _vader_file_rawagg(fp, tag, variant, band=band, sia=sia)
            part.to_parquet(cp, index=False)
        parts.append(part)

    agg = _reduce_parts(parts)
    if agg.empty:
        raise ValueError("VADER: пустой результат")

    g = agg.copy()
    g["vader_news_count"] = g["n"].round().astype("int32")
    g["vader_score_mean"] = np.where(g["n"] > 0, g["ssum"] / g["n"], np.nan).astype("float32")
    g["vader_neg_share"] = np.where(g["n"] > 0, g["nneg"] / g["n"], np.nan).astype("float32")
    g["vader_pos_share"] = np.where(g["n"] > 0, g["npos"] / g["n"], np.nan).astype("float32")
    g["vader_neu_share"] = (1.0 - g["vader_neg_share"] - g["vader_pos_share"]).clip(0.0, 1.0).astype("float32")
    g["vader_active_score_mean"] = g["vader_score_mean"]
    g = g.drop(columns=["n", "ssum", "nneg", "npos"])

    out_path = os.path.join(OUTPUT_DIR, "daily_news_vader_2019_2023.parquet")
    g.to_parquet(out_path, index=False)
    print("Сохранено:", out_path, "Строк:", f"{len(g):,}")
    return g

def _textblob_file_rawagg(file_path: str, source_tag: str, variant: str,
                          band: float) -> pd.DataFrame:
    parts = []
    buffer = []
    reader = pd.read_csv(file_path, chunksize=CHUNK_SIZE, low_memory=False)

    def _tb(text: str) -> float:
        try:
            return float(TextBlob(text).sentiment.polarity)
        except Exception:
            return 0.0

    for raw in tqdm(reader, desc=f"textblob: {source_tag}"):
        ch = _prep_news_chunk_variant(raw, source_tag, variant)
        if ch.empty:
            continue

        texts = ch["text"].astype(str).tolist()
        scores = np.fromiter((_tb(x) for x in texts), dtype=np.float32, count=len(texts))
        neg = (scores <= -float(band)).astype(np.int32)
        pos = (scores >= float(band)).astype(np.int32)

        tmp = ch[["ticker", "day"]].copy()
        tmp = tmp.rename(columns={"day": "date"})
        tmp["n"] = 1
        tmp["ssum"] = scores.astype(np.float32)
        tmp["nneg"] = neg
        tmp["npos"] = pos

        g = tmp.groupby(["ticker", "date"], sort=False).sum(numeric_only=True).reset_index()
        buffer.append(g)

        if len(buffer) >= 25:
            parts.append(_reduce_parts(buffer))
            buffer = []

    if buffer:
        parts.append(_reduce_parts(buffer))

    out = _reduce_parts(parts)
    return out

def build_textblob_daily(band: float = 0.05) -> pd.DataFrame:
    parts = []
    for tag, fp in NEWS_SOURCES.items():
        variant = selected_variants[tag]

        cp = _checkpoint_path("textblob", tag, variant)
        if RESUME_FROM_CHECKPOINTS and os.path.exists(cp) and not FORCE_RECOMPUTE_CHECKPOINTS:
            part = pd.read_parquet(cp)
        else:
            part = _textblob_file_rawagg(fp, tag, variant, band=band)
            part.to_parquet(cp, index=False)
        parts.append(part)

    agg = _reduce_parts(parts)
    if agg.empty:
        raise ValueError("TextBlob: пустой результат")

    g = agg.copy()
    g["textblob_news_count"] = g["n"].round().astype("int32")
    g["textblob_score_mean"] = np.where(g["n"] > 0, g["ssum"] / g["n"], np.nan).astype("float32")
    g["textblob_neg_share"] = np.where(g["n"] > 0, g["nneg"] / g["n"], np.nan).astype("float32")
    g["textblob_pos_share"] = np.where(g["n"] > 0, g["npos"] / g["n"], np.nan).astype("float32")
    g["textblob_neu_share"] = (1.0 - g["textblob_neg_share"] - g["textblob_pos_share"]).clip(0.0, 1.0).astype("float32")
    g["textblob_active_score_mean"] = g["textblob_score_mean"]
    g = g.drop(columns=["n", "ssum", "nneg", "npos"])

    out_path = os.path.join(OUTPUT_DIR, "daily_news_textblob_2019_2023.parquet")
    g.to_parquet(out_path, index=False)
    print("Сохранено:", out_path, "Строк:", f"{len(g):,}")
    return g

def _hf_file_rawagg(file_path: str, source_tag: str, variant: str,
                    tok, mdl, neg_i, neu_i, pos_i,
                    batch_size: int = None, max_length: int = None,
                    prefix: str = "hf") -> pd.DataFrame:
    batch_size = int(HF_BATCH_SIZE if batch_size is None else batch_size)
    max_length = int(HF_MAX_LENGTH if max_length is None else max_length)

    parts = []
    buffer = []
    reader = pd.read_csv(file_path, chunksize=CHUNK_SIZE, low_memory=False)

    for raw in tqdm(reader, desc=f"{prefix}: {source_tag}"):
        ch = _prep_news_chunk_variant(raw, source_tag, variant)
        if ch.empty:
            continue

        texts = ch["text"].astype(str).tolist()
        score, abs_score, pneg, pneu, ppos = _hf_predict(
            texts, tok, mdl, neg_i, neu_i, pos_i,
            batch_size=batch_size, max_length=max_length
        )

        mass = (ppos + pneg).astype(np.float32)
        active = (score / (mass + 1e-9)).astype(np.float32)
        pneg_a = (pneg / (mass + 1e-9)).astype(np.float32)
        ppos_a = (ppos / (mass + 1e-9)).astype(np.float32)

        tmp = ch[["ticker", "day"]].copy()
        tmp = tmp.rename(columns={"day": "date"})
        tmp["n"] = 1
        tmp["ssum"] = score.astype(np.float32)
        tmp["asum"] = active.astype(np.float32)
        tmp["neg"] = pneg.astype(np.float32)
        tmp["neu"] = pneu.astype(np.float32)
        tmp["pos"] = ppos.astype(np.float32)
        tmp["neg_a"] = pneg_a.astype(np.float32)
        tmp["pos_a"] = ppos_a.astype(np.float32)

        g = tmp.groupby(["ticker", "date"], sort=False).sum(numeric_only=True).reset_index()
        buffer.append(g)

        if len(buffer) >= 25:
            parts.append(_reduce_parts(buffer))
            buffer = []

    if buffer:
        parts.append(_reduce_parts(buffer))

    out = _reduce_parts(parts)
    return out

def build_hf_daily(model_id: str, prefix: str,
                   batch_size: int = None, max_length: int = None) -> pd.DataFrame:
    tok, mdl, id2lab, lab2id = _load_hf_model(model_id)
    neg_i, neu_i, pos_i = _resolve_sentiment_slots(id2lab, lab2id)

    parts = []
    for tag, fp in NEWS_SOURCES.items():
        variant = selected_variants[tag]

        cp = _checkpoint_path(prefix, tag, variant)
        if RESUME_FROM_CHECKPOINTS and os.path.exists(cp) and not FORCE_RECOMPUTE_CHECKPOINTS:
            part = pd.read_parquet(cp)
        else:
            part = _hf_file_rawagg(
                fp, tag, variant,
                tok, mdl, neg_i, neu_i, pos_i,
                batch_size=batch_size, max_length=max_length,
                prefix=prefix
            )
            part.to_parquet(cp, index=False)
        parts.append(part)

    agg = _reduce_parts(parts)
    if agg.empty:
        raise ValueError(f"{prefix}: пустой результат")

    g = agg.copy()
    g[f"{prefix}_news_count"] = g["n"].round().astype("int32")
    g[f"{prefix}_score_mean"] = np.where(g["n"] > 0, g["ssum"] / g["n"], np.nan).astype("float32")
    g[f"{prefix}_neg_share"] = np.where(g["n"] > 0, g["neg"] / g["n"], np.nan).astype("float32")
    g[f"{prefix}_neu_share"] = np.where(g["n"] > 0, g["neu"] / g["n"], np.nan).astype("float32")
    g[f"{prefix}_pos_share"] = np.where(g["n"] > 0, g["pos"] / g["n"], np.nan).astype("float32")

    g[f"{prefix}_active_score_mean"] = np.where(g["n"] > 0, g["asum"] / g["n"], np.nan).astype("float32")
    g[f"{prefix}_active_neg_share"] = np.where(g["n"] > 0, g["neg_a"] / g["n"], np.nan).astype("float32")
    g[f"{prefix}_active_pos_share"] = np.where(g["n"] > 0, g["pos_a"] / g["n"], np.nan).astype("float32")

    g = g.drop(columns=["n", "ssum", "asum", "neg", "neu", "pos", "neg_a", "pos_a"])

    out_path = os.path.join(OUTPUT_DIR, f"daily_news_{prefix}_2019_2023.parquet")
    g.to_parquet(out_path, index=False)
    print("Сохранено:", out_path, "Строк:", f"{len(g):,}")

    del mdl
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()

    return g

Выбранные варианты: {'nasdaq': 'title_article'}


## Блок 8. Загрузка модельных результатов

In [9]:
BASE_DEVICE = DEVICE

def _cleanup_torch_device(dev: str) -> None:
    try:
        if dev == "cuda" and torch.cuda.is_available():
            torch.cuda.empty_cache()
        elif dev == "mps" and torch.backends.mps.is_available():
            torch.mps.empty_cache()
    except Exception:
        pass
    gc.collect()

def _set_cpu_runtime() -> None:
    try:
        torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))
    except Exception:
        pass
    try:
        torch.set_num_interop_threads(1)
    except Exception:
        pass

def _load_hf_model(model_id: str):
    exec_device = str(DEVICE)

    tok = AutoTokenizer.from_pretrained(
        model_id,
        use_fast=True,
        local_files_only=True,
    )

    if exec_device == "cpu":
        _set_cpu_runtime()
        mdl = AutoModelForSequenceClassification.from_pretrained(
            model_id,
            local_files_only=True,
        )
    else:
        mdl = AutoModelForSequenceClassification.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            local_files_only=True,
        )

    mdl = mdl.to(exec_device)
    mdl.eval()

    raw = getattr(mdl.config, "id2label", {}) or {}
    id2lab = {int(k): str(v).lower() for k, v in raw.items()} if isinstance(raw, dict) else {}

    raw2 = getattr(mdl.config, "label2id", {}) or {}
    lab2id = {str(k).lower(): int(v) for k, v in raw2.items()} if isinstance(raw2, dict) else {}

    return tok, mdl, id2lab, lab2id

print("Базовое устройство:", BASE_DEVICE)

def _meta_path(out_path: str) -> str:
    return out_path + ".meta.json"

def load_or_build_daily(model_tag: str, builder_fn, out_path: str, meta: dict) -> pd.DataFrame:
    meta_file = _meta_path(out_path)

    if os.path.exists(out_path) and SKIP_IF_DAILY_EXISTS and not FORCE_REBUILD_DAILY:
        df = pd.read_parquet(out_path)
        print(f"[{model_tag}] загружен готовый файл: {out_path} | строк: {len(df):,}")
        return df

    if not ALLOW_MODEL_REBUILD:
        raise FileNotFoundError(
            f"[{model_tag}] Готовый файл не найден: {out_path}. Пересборка отключена, чтобы не перезапускать модели."
        )

    print(f"[{model_tag}] запуск или возобновление из чекпойнтов...")
    df = builder_fn()
    df.to_parquet(out_path, index=False)

    with open(meta_file, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    print(f"[{model_tag}] сохранено: {out_path}")
    return df

def _build_hf_daily_on_device(model_id: str, prefix: str,
                              batch_size: int,
                              max_length: int,
                              exec_device: str) -> pd.DataFrame:
    global DEVICE

    prev_device = str(DEVICE)
    target_device = str(exec_device)

    if target_device != prev_device:
        _cleanup_torch_device(prev_device)

    DEVICE = target_device
    try:
        print(f"[{prefix}] устройство={target_device} | batch_size={batch_size} | max_length={max_length}")
        return build_hf_daily(
            model_id=model_id,
            prefix=prefix,
            batch_size=int(batch_size),
            max_length=int(max_length),
        )
    finally:
        _cleanup_torch_device(target_device)
        DEVICE = prev_device

def _is_retryable_hf_error(e: Exception) -> bool:
    msg = str(e).lower()
    flags = [
        "out of memory",
        "mps",
        "metal",
        "placeholder storage",
        "not implemented",
        "allocate",
    ]
    return any(x in msg for x in flags)

def _run_hf_with_plan(model_id: str, prefix: str, plan, max_length: int) -> pd.DataFrame:
    last_err = None

    for exec_device, batch_size in plan:
        try:
            return _build_hf_daily_on_device(
                model_id=model_id,
                prefix=prefix,
                batch_size=batch_size,
                max_length=max_length,
                exec_device=exec_device,
            )
        except Exception as e:
            last_err = e
            print(f"[{prefix}] ошибка на {exec_device} при batch_size={batch_size}: {e}")
            if not _is_retryable_hf_error(e):
                raise

    if last_err is not None:
        raise last_err
    raise RuntimeError(f"{prefix}: нет подходящего плана выполнения")

if BASE_DEVICE == "mps":
    HF_RUNTIME = {
        "finbert": {"plan": [("mps", min(24, int(HF_BATCH_SIZE))), ("mps", 16), ("cpu", 8)], "max_length": int(HF_MAX_LENGTH)},
        "finbert_tone": {"plan": [("mps", min(24, int(HF_BATCH_SIZE))), ("mps", 16), ("cpu", 8)], "max_length": int(HF_MAX_LENGTH)},
        "finroberta": {"plan": [("mps", 24), ("mps", 16), ("cpu", 4)], "max_length": 192},
        "distilroberta_finnews": {"plan": [("mps", 48), ("mps", 32), ("cpu", 12)], "max_length": 192},
        "deberta_finnews": {"plan": [("mps", 16), ("mps", 8), ("cpu", 4)], "max_length": 192},
        "twroberta": {"plan": [("mps", 48), ("mps", 32), ("cpu", 12)], "max_length": 192},
    }
else:
    HF_RUNTIME = {
        "finbert": {"plan": [("cpu", 8)], "max_length": int(HF_MAX_LENGTH)},
        "finbert_tone": {"plan": [("cpu", 8)], "max_length": int(HF_MAX_LENGTH)},
        "finroberta": {"plan": [("cpu", 4)], "max_length": 192},
        "distilroberta_finnews": {"plan": [("cpu", 12)], "max_length": 192},
        "deberta_finnews": {"plan": [("cpu", 4)], "max_length": 192},
        "twroberta": {"plan": [("cpu", 12)], "max_length": 192},
    }

print("Планы выполнения:")
for k, v in HF_RUNTIME.items():
    print(k, "->", v)

Базовое устройство: mps
Планы выполнения:
finbert -> {'plan': [('mps', 24), ('mps', 16), ('cpu', 8)], 'max_length': 512}
finbert_tone -> {'plan': [('mps', 24), ('mps', 16), ('cpu', 8)], 'max_length': 512}
finroberta -> {'plan': [('mps', 24), ('mps', 16), ('cpu', 4)], 'max_length': 192}
distilroberta_finnews -> {'plan': [('mps', 48), ('mps', 32), ('cpu', 12)], 'max_length': 192}
deberta_finnews -> {'plan': [('mps', 16), ('mps', 8), ('cpu', 4)], 'max_length': 192}
twroberta -> {'plan': [('mps', 48), ('mps', 32), ('cpu', 12)], 'max_length': 192}


## Блок 9. Дневные агрегаты тональности

Загружаем или, если это явно разрешено, собираем ежедневные сентимент-таблицы по 8ми моделям

In [10]:
print_step("Загрузка или сборка ежедневных агрегатов тональности по 8 моделям")

max_news = None if (MAX_NEWS_PER_DAY is None or int(MAX_NEWS_PER_DAY) <= 0) else int(MAX_NEWS_PER_DAY)
date_range = [str(START_DATE.date()), str(END_DATE.date())]

specs = [
    (
        "vader",
        lambda: build_vader_daily(band=0.05),
        {
            "model_tag": "vader",
            "run_tag": RUN_TAG,
            "date_range": date_range,
            "hf_max_length": None,
            "max_news_per_day": max_news,
            "model_id": None,
        },
    ),
    (
        "textblob",
        lambda: build_textblob_daily(band=0.05),
        {
            "model_tag": "textblob",
            "run_tag": RUN_TAG,
            "date_range": date_range,
            "hf_max_length": None,
            "max_news_per_day": max_news,
            "model_id": None,
        },
    ),
    (
        "finbert",
        lambda: _run_hf_with_plan(
            model_id="ProsusAI/finbert",
            prefix="finbert",
            plan=HF_RUNTIME["finbert"]["plan"],
            max_length=HF_RUNTIME["finbert"]["max_length"],
        ),
        {
            "model_tag": "finbert",
            "run_tag": RUN_TAG,
            "date_range": date_range,
            "hf_max_length": HF_RUNTIME["finbert"]["max_length"],
            "max_news_per_day": max_news,
            "model_id": "ProsusAI/finbert",
        },
    ),
    (
        "finbert_tone",
        lambda: _run_hf_with_plan(
            model_id="yiyanghkust/finbert-tone",
            prefix="finbert_tone",
            plan=HF_RUNTIME["finbert_tone"]["plan"],
            max_length=HF_RUNTIME["finbert_tone"]["max_length"],
        ),
        {
            "model_tag": "finbert_tone",
            "run_tag": RUN_TAG,
            "date_range": date_range,
            "hf_max_length": HF_RUNTIME["finbert_tone"]["max_length"],
            "max_news_per_day": max_news,
            "model_id": "yiyanghkust/finbert-tone",
        },
    ),
    (
        "finroberta",
        lambda: _run_hf_with_plan(
            model_id="soleimanian/financial-roberta-large-sentiment",
            prefix="finroberta",
            plan=HF_RUNTIME["finroberta"]["plan"],
            max_length=HF_RUNTIME["finroberta"]["max_length"],
        ),
        {
            "model_tag": "finroberta",
            "run_tag": RUN_TAG,
            "date_range": date_range,
            "hf_max_length": HF_RUNTIME["finroberta"]["max_length"],
            "max_news_per_day": max_news,
            "model_id": "soleimanian/financial-roberta-large-sentiment",
        },
    ),
    (
        "distilroberta_finnews",
        lambda: _run_hf_with_plan(
            model_id="mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis",
            prefix="distilroberta_finnews",
            plan=HF_RUNTIME["distilroberta_finnews"]["plan"],
            max_length=HF_RUNTIME["distilroberta_finnews"]["max_length"],
        ),
        {
            "model_tag": "distilroberta_finnews",
            "run_tag": RUN_TAG,
            "date_range": date_range,
            "hf_max_length": HF_RUNTIME["distilroberta_finnews"]["max_length"],
            "max_news_per_day": max_news,
            "model_id": "mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis",
        },
    ),
    (
        "deberta_finnews",
        lambda: _run_hf_with_plan(
            model_id="mrm8488/deberta-v3-ft-financial-news-sentiment-analysis",
            prefix="deberta_finnews",
            plan=HF_RUNTIME["deberta_finnews"]["plan"],
            max_length=HF_RUNTIME["deberta_finnews"]["max_length"],
        ),
        {
            "model_tag": "deberta_finnews",
            "run_tag": RUN_TAG,
            "date_range": date_range,
            "hf_max_length": HF_RUNTIME["deberta_finnews"]["max_length"],
            "max_news_per_day": max_news,
            "model_id": "mrm8488/deberta-v3-ft-financial-news-sentiment-analysis",
        },
    ),
    (
        "twroberta",
        lambda: _run_hf_with_plan(
            model_id="cardiffnlp/twitter-roberta-base-sentiment-latest",
            prefix="twroberta",
            plan=HF_RUNTIME["twroberta"]["plan"],
            max_length=HF_RUNTIME["twroberta"]["max_length"],
        ),
        {
            "model_tag": "twroberta",
            "run_tag": RUN_TAG,
            "date_range": date_range,
            "hf_max_length": HF_RUNTIME["twroberta"]["max_length"],
            "max_news_per_day": max_news,
            "model_id": "cardiffnlp/twitter-roberta-base-sentiment-latest",
        },
    ),
]

model_map = {}

for tag, fn, meta in specs:
    out_path = os.path.join(OUTPUT_DIR, f"daily_news_{tag}_2019_2023.parquet")
    model_map[tag] = load_or_build_daily(tag, fn, out_path, meta)

model_map["finbert"] = model_map["finbert"].copy()
fin_nc = "finbert_news_count"
if fin_nc in model_map["finbert"].columns:
    model_map["finbert"]["news_count"] = model_map["finbert"][fin_nc]
else:
    raise ValueError("Колонка finbert_news_count не найдена в выходе FinBERT")

display(model_map["finbert"].head())


Загрузка или сборка ежедневных агрегатов тональности по 8 моделям
[vader] загружен готовый файл: outputs_01/daily_news_vader_2019_2023.parquet | строк: 567,418
[textblob] загружен готовый файл: outputs_01/daily_news_textblob_2019_2023.parquet | строк: 567,418
[finbert] загружен готовый файл: outputs_01/daily_news_finbert_2019_2023.parquet | строк: 567,418
[finbert_tone] загружен готовый файл: outputs_01/daily_news_finbert_tone_2019_2023.parquet | строк: 567,418
[finroberta] загружен готовый файл: outputs_01/daily_news_finroberta_2019_2023.parquet | строк: 567,418
[distilroberta_finnews] загружен готовый файл: outputs_01/daily_news_distilroberta_finnews_2019_2023.parquet | строк: 567,418
[deberta_finnews] загружен готовый файл: outputs_01/daily_news_deberta_finnews_2019_2023.parquet | строк: 567,418
[twroberta] загружен готовый файл: outputs_01/daily_news_twroberta_2019_2023.parquet | строк: 567,418


,ticker,date,finbert_news_count,finbert_score_mean,finbert_neg_share,finbert_neu_share,finbert_pos_share,finbert_active_score_mean,finbert_active_neg_share,finbert_active_pos_share,news_count
0,A,2019-01-03,2,0.570240,0.030882,0.367996,0.601122,0.908074,0.045963,0.954037,2
1,A,2019-01-09,1,0.877619,0.009204,0.103973,0.886823,0.979456,0.010272,0.989728,1
2,A,2019-01-14,1,0.657978,0.022156,0.297710,0.680134,0.936903,0.031549,0.968451,1
3,A,2019-01-18,1,0.001846,0.035643,0.926869,0.037489,0.025239,0.487380,0.512620,1
4,A,2019-01-19,1,-0.088948,0.129746,0.829455,0.040798,-0.521554,0.760777,0.239223,1


## Блок 10. Привязка к торговым дням

Выравниваем дневные агрегаты по торговому календарю каждой компании

In [11]:
def align_daily_to_trade_per_ticker(daily_df: pd.DataFrame, day_col: str = "date") -> pd.DataFrame:
    a = daily_df.copy()
    a["ticker"] = a["ticker"].astype(str).str.strip()
    a[day_col] = pd.to_datetime(a[day_col], errors="coerce").dt.normalize()
    a = a.dropna(subset=["ticker", day_col])
    a = a.sort_values(["ticker", day_col], kind="mergesort").reset_index(drop=True)

    if "news_count" in a.columns:
        count_col = "news_count"
    else:
        nc = [c for c in a.columns if c.endswith("_news_count")]
        count_col = nc[0] if nc else None

    if count_col is None:
        count_col = "_tmp_news_count"
        a[count_col] = 1

    out_parts = []
    for tic, g in a.groupby("ticker", sort=False):
        tcal = trade.loc[trade["ticker"] == tic, ["trade_date"]].sort_values("trade_date")
        if tcal.empty:
            continue

        merged = pd.merge_asof(
            g.sort_values(day_col, kind="mergesort"),
            tcal,
            left_on=day_col,
            right_on="trade_date",
            direction="forward",
            allow_exact_matches=True,
            tolerance=pd.Timedelta("7D"),
        )

        merged = merged.dropna(subset=["trade_date"]).copy()
        if merged.empty:
            continue

        merged = merged.drop(columns=[day_col]).rename(columns={"trade_date": "date"})
        merged["date"] = pd.to_datetime(merged["date"]).dt.normalize()

        num_cols = [
            c for c in merged.columns
            if c not in ["ticker", "date", count_col] and pd.api.types.is_numeric_dtype(merged[c])
        ]

        w = pd.to_numeric(merged[count_col], errors="coerce").fillna(0).astype(float)

        tmp = merged[["ticker", "date"] + num_cols].copy()
        tmp[count_col] = w.values

        do_not_weight = {count_col}
        do_not_weight.update([c for c in num_cols if c.endswith("_news_count")])
        weighted_cols = [c for c in num_cols if c not in do_not_weight]

        for c in weighted_cols:
            tmp[c] = pd.to_numeric(tmp[c], errors="coerce").astype(float) * tmp[count_col]

        agg = tmp.groupby(["ticker", "date"], sort=False).sum(numeric_only=True)

        denom = agg[count_col].replace(0, np.nan)
        for c in weighted_cols:
            agg[c] = agg[c] / denom

        agg[count_col] = agg[count_col].fillna(0).round().astype("int32")
        out_parts.append(agg.reset_index())

    out = pd.concat(out_parts, ignore_index=True) if out_parts else a.iloc[0:0].copy()
    out = out.sort_values(["ticker", "date"], kind="mergesort").reset_index(drop=True)

    if "_tmp_news_count" in out.columns:
        out = out.drop(columns=["_tmp_news_count"])
    return out

print_step("Привязка дневных сентимент-агрегатов к торговым дням")

aligned = {}
for tag, df in model_map.items():
    aligned[tag] = align_daily_to_trade_per_ticker(df, day_col="date")
    dup = aligned[tag].duplicated(["ticker", "date"]).sum()
    if dup:
        raise ValueError(f"{tag}: дубликаты (ticker,date) после выравнивания: {dup}")

print("Строк после выравнивания:")
for tag, df in aligned.items():
    print(f"  - {tag}: {len(df):,}")


Привязка дневных сентимент-агрегатов к торговым дням
Строк после выравнивания:
  - vader: 534,585
  - textblob: 534,585
  - finbert: 534,585
  - finbert_tone: 534,585
  - finroberta: 534,585
  - distilroberta_finnews: 534,585
  - deberta_finnews: 534,585
  - twroberta: 534,585


## Блок 11. Итоговая таблица

Объединяем доходности, таргеты и сентимент-признаки в единую таблицу

In [12]:
print_step("Сборка итоговой таблицы")

master = ret.copy()

master = master.merge(aligned["finbert"], on=["ticker", "date"], how="left")

for tag in [k for k in aligned.keys() if k != "finbert"]:
    cols = [c for c in aligned[tag].columns if c not in ["ticker", "date"]]
    master = master.merge(aligned[tag][["ticker", "date"] + cols], on=["ticker", "date"], how="left")

if "sector" in companies.columns:
    master = master.merge(companies[["ticker", "sector"]], on="ticker", how="left")

master["news_n"] = pd.to_numeric(master.get("news_count"), errors="coerce").fillna(0).astype("int32")
master["has_news_today"] = (master["news_n"] > 0).astype("int8")

master = master.replace([np.inf, -np.inf], np.nan)

sent_cols = [c for c in master.columns if any(
    c.endswith(suf) for suf in [
        "_score_mean", "_active_score_mean",
        "_neg_share", "_neu_share", "_pos_share",
        "_active_neg_share", "_active_pos_share"
    ]
)]

mask_no_news = master["has_news_today"] == 0
master.loc[mask_no_news, sent_cols] = master.loc[mask_no_news, sent_cols].fillna(0.0)

master = master.sort_values(["ticker", "date"], kind="mergesort").reset_index(drop=True)

print(f"Итоговая таблица: строк {len(master):,}, тикеров {master['ticker'].nunique()}, колонок {master.shape[1]}")
print(f"Период: {master['date'].min().date()} — {master['date'].max().date()}")

display(master.head())


Сборка итоговой таблицы
Итоговая таблица: строк 2,580,236, тикеров 2054, колонок 76
Период: 2019-01-02 — 2023-12-28


,ticker,date,price,ret_log,volume,price_col_used,mkt_ret_log,excess_ret_log,y1_ex,y2_ex,y3_ex,y5_ex,finbert_news_count,finbert_score_mean,finbert_neg_share,finbert_neu_share,finbert_pos_share,finbert_active_score_mean,finbert_active_neg_share,finbert_active_pos_share,news_count,vader_score_mean,vader_neg_share,vader_pos_share,vader_neu_share,vader_active_score_mean,vader_news_count,textblob_score_mean,textblob_neg_share,textblob_pos_share,textblob_neu_share,textblob_active_score_mean,textblob_news_count,finbert_tone_score_mean,finbert_tone_neg_share,finbert_tone_neu_share,finbert_tone_pos_share,finbert_tone_active_score_mean,finbert_tone_active_neg_share,finbert_tone_active_pos_share,finbert_tone_news_count,finroberta_score_mean,finroberta_neg_share,finroberta_neu_share,finroberta_pos_share,finroberta_active_score_mean,finroberta_active_neg_share,finroberta_active_pos_share,finroberta_news_count,distilroberta_finnews_score_mean,distilroberta_finnews_neg_share,distilroberta_finnews_neu_share,distilroberta_finnews_pos_share,distilroberta_finnews_active_score_mean,distilroberta_finnews_active_neg_share,distilroberta_finnews_active_pos_share,distilroberta_finnews_news_count,deberta_finnews_score_mean,deberta_finnews_neg_share,deberta_finnews_neu_share,deberta_finnews_pos_share,deberta_finnews_active_score_mean,deberta_finnews_active_neg_share,deberta_finnews_active_pos_share,deberta_finnews_news_count,twroberta_score_mean,twroberta_neg_share,twroberta_neu_share,twroberta_pos_share,twroberta_active_score_mean,twroberta_active_neg_share,twroberta_active_pos_share,twroberta_news_count,sector,news_n,has_news_today
0,A,2019-01-02,64.968681,NaN,2113300.0,adj close,NaN,NaN,-0.013383,-0.012302,0.000856,0.022114,NaN,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.00,0.0,0.0,0.0,0.00,NaN,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0.000000e+00,0.0,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,NaN,Healthcare,0,0
1,A,2019-01-03,62.575249,-0.037536,5383900.0,adj close,-0.024152,-0.013383,0.001081,0.014239,0.019441,0.041318,2.0,0.57024,0.030882,0.367996,0.601122,0.908074,0.045963,0.954037,2.0,0.0,0.0,0.0,1.0,0.0,2.0,0.05,0.0,0.5,0.5,0.05,2.0,1.0,4.233068e-09,5.556301e-08,1.0,1.0,4.233069e-09,1.0,2.0,0.997358,0.000689,0.001266,0.998047,0.998621,0.000689,0.999311,2.0,0.49968,0.000146,0.500072,0.499826,0.724141,0.137928,0.862069,2.0,0.998218,0.000317,0.001135,0.998535,0.999365,0.000318,0.999682,2.0,0.41049,0.007906,0.57373,0.418396,0.935005,0.032497,0.967503,2.0,Healthcare,2,1
2,A,2019-01-04,64.741203,0.034028,3123700.0,adj close,0.032947,0.001081,0.013158,0.018360,0.034416,0.046694,NaN,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.00,0.0,0.0,0.0,0.00,NaN,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0.000000e+00,0.0,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,NaN,Healthcare,0,0
3,A,2019-01-07,66.115929,0.021012,3235100.0,adj close,0.007854,0.013158,0.005202,0.021258,0.027079,0.030664,NaN,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.0,0.0,0.0,0.0,0.0,NaN,0.00,0.0,0.0,0.0,0.00,NaN,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0.000000e+00,0.0,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,NaN,Healthcare,0,0
4,A,2019-01-08,67.085175,0.014553,1578100.0,adj close,0.009351,0.005202,0.016056,0.021877,0.028334,0.026179,

## Блок 12. Проверка и сохранение

Проверяем обязательные колонки и сохраняем финальные выходы

In [13]:
print_step("Проверка и сохранение финальных таблиц")

dup_master = master.duplicated(subset=["ticker", "date"]).sum()
print("Дубликаты (ticker,date):", int(dup_master))
if dup_master:
    raise ValueError("В итоговой таблице есть дубликаты. Устраните их перед дальнейшей работой.")

required = [
    "ticker", "date", "ret_log", "mkt_ret_log", "excess_ret_log",
    "y1_ex", "y2_ex", "y3_ex", "y5_ex",
    "has_news_today", "news_n",
    "finbert_score_mean", "vader_score_mean", "textblob_score_mean",
]
missing = [c for c in required if c not in master.columns]
if missing:
    raise ValueError(f"Не хватает обязательных колонок: {missing}")

print("Все обязательные колонки присутствуют")
print("Доля дней с новостями (has_news_today):", f"{master['has_news_today'].mean()*100:.1f}%")

MASTER_PATH = os.path.join(OUTPUT_DIR, "returns_sentiment_enhanced.parquet")
master.to_parquet(MASTER_PATH, index=False)
print(f"Итоговая таблица сохранена: {MASTER_PATH}")


Проверка и сохранение финальных таблиц
Дубликаты (ticker,date): 0
Все обязательные колонки присутствуют
Доля дней с новостями (has_news_today): 20.7%
Итоговая таблица сохранена: outputs_01/returns_sentiment_enhanced.parquet


## Блок 13. Примеры исходных текстов

Собираем небольшой набор текстов для ручной проверки выбранного варианта

In [14]:
SAMPLE_N = 20

with open(SELECTED_VARIANTS_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

selected_variants = meta["selected_variants"]

src_tag = list(NEWS_SOURCES.keys())[0]
news_fp = list(NEWS_SOURCES.values())[0]
variant = selected_variants[src_tag]

print_step("Подготовка примеров сырых текстов для ручной проверки")

reader = pd.read_csv(news_fp, chunksize=CHUNK_SIZE, low_memory=False)

examples = []
for raw in reader:
    ch = _prep_news_chunk_base(raw, src_tag)
    if ch.empty:
        continue
    text = build_text_variant(ch, variant)
    m = (text != "")
    if not m.any():
        continue
    out = ch.loc[m, ["ticker", "day", "url"]].copy()
    out["text"] = text.loc[m].astype(str)
    examples.append(out)
    if sum(len(x) for x in examples) >= SAMPLE_N * 5:
        break

ex = pd.concat(examples, ignore_index=True).drop_duplicates(subset=["ticker", "day", "url"])
ex = ex.sample(n=min(SAMPLE_N, len(ex)), random_state=1).reset_index(drop=True)

show = ex.copy()
show["text"] = show["text"].astype(str).str.slice(0, 400)
display(show)


Подготовка примеров сырых текстов для ручной проверки


,ticker,day,url,text
0,AIR,2021-11-19,https://www.nasdaq.com/articles/exclusive-u.s.-house-democrats-urge-fcc-to-avoid-potential-air-safety-wireless-danger,"EXCLUSIVE-U.S. House Democrats urge FCC to avoid potential air safety wireless danger. By David Shepardson WASHINGTON, Nov 19 (Reuters) - Two key U.S. House Democrats on Friday..."
1,AKO-A,2023-12-15,https://www.nasdaq.com/articles/monday-sector-laggards%3A-rubber-plastics-general-contractors-builders,"Monday Sector Laggards: Rubber & Plastics, General Contractors & Builders. In trading on Monday, rubber & plastics shares were relative laggards, down on the day by about 2.3%...."
2,ACGLO,2023-12-11,https://www.nasdaq.com/articles/seeking-passive-income-7-dividend-stocks-you-dont-want-to-miss.,"Seeking Passive Income? 7 Dividend Stocks You Don’t Want to Miss.. InvestorPlace - Stock Market News, Stock Advice & Trading Tips The quest for the passive income from a stable..."
3,AEE,2022-07-28,https://www.nasdaq.com/articles/ameren-aee-earnings-expected-to-grow%3A-what-to-know-ahead-of-next-weeks-release-0,Ameren (AEE) Earnings Expected to Grow: What to Know Ahead of Next Week's Release. Ameren (AEE) is expected to deliver a year-over-year increase in earnings on higher revenues ...
4,AKBA,2023-06-02,https://www.nasdaq.com/articles/all-you-need-to-know-about-akebia-therapeutics-akba-rating-upgrade-to-buy,All You Need to Know About Akebia Therapeutics (AKBA) Rating Upgrade to Buy. Akebia Therapeutics (AKBA) could be a solid choice for investors given its recent upgrade to a Zack...
5,AKO-A,2023-12-16,https://www.nasdaq.com/articles/heres-why-eog-resources-eog-is-a-solid-investment-bet,"Here's Why EOG Resources (EOG) is a Solid Investment Bet. EOG Resources, Inc. EOG has witnessed upward earnings estimate revisions for 2023 and 2024 over the past 60 days. Fact..."
6,AAPL,2023-04-04,https://www.nasdaq.com/articles/the-7-best-forever-stocks-to-buy-for-april-2023,"The 7 Best Forever Stocks to Buy for April 2023. InvestorPlace - Stock Market News, Stock Advice & Trading Tips The good thing about forever stocks is you only have to buy them..."
7,AMD,2022-07-22,https://www.nasdaq.com/articles/heres-why-intel-stock-could-be-in-a-soup-this-earnings-season,Here's Why Intel Stock Could Be in a Soup This Earnings Season. Chip giant Intel's (NASDAQ: INTC) terrible year may be about to get worse when the company releases its second-q...
8,AAPL,2022-07-27,https://www.nasdaq.com/articles/lg-display-reverses-into-loss-as-inflation-ends-covid-fuelled-demand-surge,"LG Display reverses into loss as inflation ends COVID-fuelled demand surge. Adds revenue, price data SEOUL, July 27 (Reuters) - South Korean display panel maker LG Display Co 0..."
9,AMC,2020-11-16,https://www.nasdaq.com/articles/universal-cinemark-agree-on-earlier-home-release-for-movies-2020-11-16,"Universal, Cinemark agree on earlier home release for movies. LOS ANGELES, Nov 16 (Reuters) - Comcast Corp's CMCSA.O Universal Pictures and Cinemark Holdings Inc CNK.N have rea..."


## Блок 14. Пример для TSLA

Формируем пример из итоговой таблицы и сохраняем его в CSV

In [15]:
master = pd.read_parquet(MASTER_PATH)

show_cols = [
    "ticker", "date", "price", "ret_log", "excess_ret_log",
    "y1_ex",
    "news_n", "has_news_today",
    "finbert_score_mean", "finbert_neg_share", "finbert_pos_share",
    "vader_score_mean",
    "textblob_score_mean",
    "sector",
]

sample = (
    master
    .loc[
        (master["ticker"] == "TSLA")
        & (master["date"] >= "2023-10-16")
        & (master["date"] <= "2023-10-23"),
        show_cols,
    ]
    .sort_values("date")
    .reset_index(drop=True)
)

num_cols = sample.select_dtypes("number").columns
sample[num_cols] = sample[num_cols].round(4)

SAMPLE_EXPORT_PATH = "master_sample_TSLA_oct2023.csv"
sample.to_csv(SAMPLE_EXPORT_PATH, index=False)

print(f"Итоговая таблица: {master.shape[0]:,} строк × {master.shape[1]} колонок")
print(f"Период: {master['date'].min().date()} — {master['date'].max().date()}")
print(f"Тикеров: {master['ticker'].nunique()}")
print(f"Сохранено: {SAMPLE_EXPORT_PATH}")
print()
print(f"Строк: {master.shape[0]:,}")
print(f"Колонок: {master.shape[1]}")
print(f"Тикеров: {master['ticker'].nunique():,}")
print(f"Период: {master['date'].min().date()} — {master['date'].max().date()}")

sample

Итоговая таблица: 2,580,236 строк × 76 колонок
Период: 2019-01-02 — 2023-12-28
Тикеров: 2054
Сохранено: master_sample_TSLA_oct2023.csv

Строк: 2,580,236
Колонок: 76
Тикеров: 2,054
Период: 2019-01-02 — 2023-12-28


,ticker,date,price,ret_log,excess_ret_log,y1_ex,news_n,has_news_today,finbert_score_mean,finbert_neg_share,finbert_pos_share,vader_score_mean,textblob_score_mean,sector
0,TSLA,2023-10-16,253.92,0.0111,0.0006,0.0037,46,1,-0.0768,0.3737,0.2969,0.6982,0.1190,Consumer Cyclical
1,TSLA,2023-10-17,254.85,0.0037,0.0037,-0.0355,28,1,-0.1806,0.4262,0.2456,0.7029,0.1094,Consumer Cyclical
2,TSLA,2023-10-18,242.68,-0.0489,-0.0355,-0.0888,41,1,-0.4195,0.5652,0.1458,0.4822,0.1029,Consumer Cyclical
3,TSLA,2023-10-19,220.11,-0.0976,-0.0888,-0.0252,64,1,-0.3720,0.5971,0.2251,0.6290,0.1050,Consumer Cyclical
4,TSLA,2023-10-20,211.99,-0.0376,-0.0252,0.0022,25,1,-0.3318,0.5522,0.2203,0.4776,0.0898,Consumer Cyclical
5,TSLA,2023-10-23,212.08,0.0004,0.0022,0.0132,33,1,-0.2366,0.4424,0.2057,0.5223,0.1389,Consumer Cyclical
